# Zebrafish Locomotor Activity Analysis

Analyses one or more DanioVision-exported spreadsheets (`.xls` / `.xlsx`) from zebrafish swimming experiments.

**Workflow:** &ensp; 1. Setup → 2. Load data & configure → 3. Distance analysis → 4. Duration analysis

When multiple plates are loaded, wells are automatically prefixed (P1:, P2:, …) to keep them distinct. Each analysis produces per-well scatter plots, a group time-series with light-period shading, summary tables, and an Excel export.


In [1]:
# @title 1 · Setup (run once)
!pip install -q matplotlib ipywidgets pandas numpy scipy openpyxl xlrd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy import stats
import time, io, os, re, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    from google.colab import files as colab_files
    _COLAB = True
except ImportError:
    _COLAB = False
try:
    from IPython.display import FileLink
    _FILELINK = True
except ImportError:
    _FILELINK = False

# ─── Global state ────────────────────────────────────────────────────────────
df = None                       # Combined dataframe (all plates)
well_ids = []                   # All well IDs (prefixed if multi-plate)
plate_labels = []               # ['P1', 'P2', ...] or [] if single plate
well_group_mapping = {}         # {group_name: [well_id, ...]}
spreadsheet_name = ""           # Combined display name
light_periods_set = set()       # pn values marked as Light

# ─── Shared helpers ──────────────────────────────────────────────────────────
def well_to_group(well_id):
    if not isinstance(well_id, str):
        return None
    for grp, wells in well_group_mapping.items():
        if grp == "Unassigned":
            continue
        for w in wells:
            if well_id == w:
                return grp
            # Fallback: strip -N suffix on both sides
            if well_id.rsplit('-', 1)[0] == w.rsplit('-', 1)[0]:
                return grp
    return None

def ci95_t(series):
    n = series.count()
    if n < 2:
        return 0.0
    se = series.std() / np.sqrt(n)
    return stats.t.ppf(0.975, df=n - 1) * se

def parse_period_spec(text):
    result = set()
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            # Handle P1:A01-1 style names — only treat as range if both sides are digits
            lo, hi = part.split("-", 1)
            if lo.strip().isdigit() and hi.strip().isdigit():
                result.update(range(int(lo.strip()), int(hi.strip()) + 1))
            else:
                # Not a numeric range, skip (shouldn't happen for period specs)
                result.add(int(part))
        else:
            result.add(int(part))
    return result

def read_spreadsheet(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.xlsx':
        return pd.read_excel(path, sheet_name=0, engine='openpyxl')
    try:
        return pd.read_excel(path, sheet_name=0, engine='xlrd')
    except Exception:
        pass
    return pd.read_csv(path, sep='\t', encoding='utf-16')

print("✓ Setup complete — run the next cell to load data.")


✓ Setup complete — run the next cell to load data.


In [3]:
# @title 2 · Load data, define groups & light periods

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 — Load Data (supports multiple plates)                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
upload_widget = widgets.FileUpload(accept='.xlsx,.xls', multiple=True)
filepath_input = widgets.Textarea(
    value="", description="Or paths:",
    placeholder="One file per line, e.g.:\n/path/to/plate1.xls\n/path/to/plate2.xls",
    layout=widgets.Layout(width="500px", height="60px"))
spreadsheet_name_input = widgets.Text(
    value="", description="Name:",
    placeholder="(defaults to filename(s))",
    layout=widgets.Layout(width="400px"))
load_button = widgets.Button(description="Load spreadsheet(s)",
                              layout=widgets.Layout(width="200px"),
                              button_style="primary")
file_summary_output = widgets.Output()

def on_load_clicked(b):
    global df, well_ids, spreadsheet_name, plate_labels
    with file_summary_output:
        clear_output()
        display(HTML("<i>Loading…</i>"))

    file_list = []  # list of (label, path, display_name)

    # Option A: file paths typed directly (one per line)
    if filepath_input.value.strip():
        paths = [p.strip() for p in filepath_input.value.strip().splitlines() if p.strip()]
        for i, p in enumerate(paths):
            if not os.path.isfile(p):
                with file_summary_output:
                    clear_output()
                    display(HTML(f"<b style='color:red;'>⚠ File not found: {p}</b>"))
                return
            label = f"P{i+1}" if len(paths) > 1 else ""
            file_list.append((label, p, os.path.basename(p)))

    # Option B: widget upload
    elif upload_widget.value:
        try:
            uploaded = upload_widget.value
            # Normalise to a list of (name, content) regardless of ipywidgets version
            items = []
            if isinstance(uploaded, dict):
                # ipywidgets 7.x
                for info in uploaded.values():
                    items.append((info["metadata"]["name"], info["content"]))
            else:
                # ipywidgets 8.x — tuple/list of FileInfo
                for info in uploaded:
                    nm = getattr(info, 'name', None) or info.get('name', 'upload.xls')
                    ct = getattr(info, 'content', None) or info.get('content', b'')
                    items.append((nm, ct))

            items.sort(key=lambda x: x[0])  # alphabetical → deterministic plate order
            for i, (fname, content) in enumerate(items):
                fpath = f"/tmp/plate_{i+1}_{fname}"
                with open(fpath, "wb") as f:
                    f.write(content)
                label = f"P{i+1}" if len(items) > 1 else ""
                file_list.append((label, fpath, fname))
        except Exception as e:
            with file_summary_output:
                clear_output()
                display(HTML(f"<b style='color:red;'>⚠ Upload error: {e}</b>"))
            return
    else:
        with file_summary_output:
            clear_output()
            display(HTML("<b style='color:red;'>⚠ No file — upload or enter path(s).</b>"))
        return

    # ── Read and combine all plates ──
    t0 = time.time()
    frames = []
    plate_labels = []
    all_lines = []

    for label, fpath, display_name in file_list:
        try:
            plate_df = read_spreadsheet(fpath)
        except Exception as e:
            with file_summary_output:
                clear_output()
                display(HTML(f"<b style='color:red;'>⚠ Failed to read {display_name}: {e}</b>"))
            return

        if "aname" not in plate_df.columns:
            with file_summary_output:
                clear_output()
                display(HTML(f"<b style='color:red;'>⚠ No 'aname' column in {display_name}.</b>"))
            return

        # Prefix well names if multi-plate
        if label:
            plate_df['aname'] = label + ":" + plate_df['aname'].astype(str)
            plate_df['_plate'] = label
            plate_labels.append(label)

        frames.append(plate_df)
        n_wells = plate_df['aname'].nunique()
        tag = f"<b>{label}:</b> " if label else ""
        all_lines.append(f"&ensp; {tag}<i>{display_name}</i> — {n_wells} wells, {len(plate_df)} rows")

    df = pd.concat(frames, ignore_index=True)
    well_ids[:] = sorted(df["aname"].dropna().unique().tolist())
    elapsed = time.time() - t0

    # Name
    if spreadsheet_name_input.value.strip():
        spreadsheet_name = spreadsheet_name_input.value.strip()
    elif len(file_list) == 1:
        spreadsheet_name = file_list[0][2]
    else:
        spreadsheet_name = f"{len(file_list)} plates"
    spreadsheet_name_input.value = spreadsheet_name

    # Summary
    header = f"<b>✓ Loaded {len(file_list)} file{'s' if len(file_list)>1 else ''}</b> in {elapsed:.2f}s"
    all_lines.insert(0, header)
    all_lines.append(f"&ensp; Combined: {len(well_ids)} wells · {df.shape[0]} rows · {df.shape[1]} columns")

    if 'pn' in df.columns:
        periods = sorted(df['pn'].dropna().unique())
        all_lines.append(f"&ensp; {len(periods)} periods (pn {int(min(periods))}–{int(max(periods))})")
    if 'start' in df.columns and 'end' in df.columns:
        total_sec = df['end'].max()
        per_dur = df.iloc[0]['end'] - df.iloc[0]['start']
        all_lines.append(f"&ensp; Total ≈ {total_sec:.0f}s ({total_sec/60:.1f} min) · period ≈ {per_dur:.0f}s")

    dist_cols = [c for c in ['lardist','smldist','inadist'] if c in df.columns]
    dur_cols  = [c for c in ['lardur','smldur','inadur'] if c in df.columns]
    all_lines.append(f"&ensp; Distance: {', '.join(dist_cols) or '⚠ none'} &ensp;|&ensp; Duration: {', '.join(dur_cols) or '⚠ none'}")

    # Build per-plate well grids (copy-paste ready)
    if plate_labels:
        grid_sections = []
        for label in plate_labels:
            plate_wells = sorted([w for w in well_ids if w.startswith(label + ":")])
            rows_map = {}
            for w in plate_wells:
                letter = w.split(":")[1][0]  # e.g. P1:A01-1 → A
                rows_map.setdefault(letter, []).append(w)
            section = f"<b>{label}:</b>\n"
            for letter in sorted(rows_map.keys()):
                section += ", ".join(rows_map[letter]) + ",\n"
            grid_sections.append(section)
        grid_text = "\n".join(grid_sections)
    else:
        rows_map = {}
        for w in well_ids:
            letter = w[0]
            rows_map.setdefault(letter, []).append(w)
        grid_text = ""
        for letter in sorted(rows_map.keys()):
            grid_text += ", ".join(rows_map[letter]) + ",\n"

    with file_summary_output:
        clear_output()
        display(HTML("<br>".join(all_lines)))
        display(HTML(f"<details open><summary><b>Wells — copy-paste ready</b></summary><pre style='font-size:12px; line-height:1.6;'>{grid_text}</pre></details>"))

load_button.on_click(on_load_clicked)

display(HTML("<h3>Step 1 — Load Data</h3>"))
display(HTML("<small>Upload file(s) <b>or</b> enter path(s) below (one per line). Multiple files → wells auto-prefixed P1:, P2:, …</small>"))
display(widgets.HBox([upload_widget, load_button]))
display(filepath_input)
display(spreadsheet_name_input)
display(file_summary_output)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 — Define Experimental Groups                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
group_count_input = widgets.IntText(value=3, description="Groups:",
                                     layout=widgets.Layout(width="150px"))
create_groups_button = widgets.Button(description="Create group inputs",
                                       layout=widgets.Layout(width="200px"))
group_inputs_box = widgets.VBox()

def on_create_groups_clicked(b):
    children = []
    for i in range(group_count_input.value):
        name_w = widgets.Text(value=f"Group {i+1}", description="Name:",
                              layout=widgets.Layout(width="160px"))
        wells_w = widgets.Textarea(
            value="", placeholder="Paste well IDs, e.g.: P1:A01-1, P1:A02-1  or  A01-1, A02-1",
            description="Wells:",
            layout=widgets.Layout(width="500px", height="55px"))
        children.append(widgets.HBox([name_w, wells_w]))
    group_inputs_box.children = children

create_groups_button.on_click(on_create_groups_clicked)

save_groups_button = widgets.Button(description="Save groups",
                                     layout=widgets.Layout(width="200px"),
                                     button_style="primary")
group_output = widgets.Output()

def on_save_groups_clicked(b):
    global well_group_mapping
    well_group_mapping = {}
    for hbox in group_inputs_box.children:
        name = hbox.children[0].value.strip()
        raw_text = hbox.children[1].value
        wells = [w.strip() for w in re.split(r'[,\s]+', raw_text) if w.strip()]
        well_group_mapping[name] = wells

    with group_output:
        clear_output()
        html = "<b>✓ Groups saved</b><br>"
        all_assigned = []
        for grp, ws in well_group_mapping.items():
            html += f"&ensp; <b>{grp}</b> ({len(ws)} wells): {', '.join(ws[:8])}"
            if len(ws) > 8:
                html += f" … +{len(ws)-8} more"
            html += "<br>"
            all_assigned.extend(ws)

        if well_ids:
            assigned_set = set(all_assigned)
            unassigned = [w for w in well_ids if w not in assigned_set]
            # Also check base-matching for flexibility
            unassigned_strict = []
            for w in unassigned:
                if not any(w.rsplit('-',1)[0] == a.rsplit('-',1)[0] for a in assigned_set):
                    unassigned_strict.append(w)
            if unassigned_strict:
                html += f"<br><span style='color:orange;'>⚠ Not in any group ({len(unassigned_strict)}): {', '.join(unassigned_strict[:16])}"
                if len(unassigned_strict) > 16:
                    html += f" … +{len(unassigned_strict)-16} more"
                html += "</span>"
            else:
                html += "<br><span style='color:green;'>✓ All wells assigned.</span>"
        display(HTML(html))

save_groups_button.on_click(on_save_groups_clicked)

display(HTML("<h3>Step 2 — Define Groups</h3>"))
display(HTML("<small>Paste well IDs from the grid above. Comma-separated, space-separated, or one-per-line all work.</small>"))
display(widgets.HBox([group_count_input, create_groups_button]))
display(group_inputs_box)
display(save_groups_button)
display(group_output)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 — Light Periods                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
light_input = widgets.Text(
    value="3-7, 13-17", description="Light pn:",
    placeholder="e.g. 3-7, 13-17",
    layout=widgets.Layout(width="500px"))
set_light_button = widgets.Button(description="Set light periods",
                                    layout=widgets.Layout(width="200px"),
                                    button_style="primary")
light_output = widgets.Output()

def on_set_light(b):
    global light_periods_set
    with light_output:
        clear_output()
    txt = light_input.value.strip()
    if not txt:
        light_periods_set = set()
        with light_output:
            display(HTML("<b>✓</b> No light periods — all periods treated as one phase."))
        return
    try:
        light_periods_set = parse_period_spec(txt)
    except Exception:
        with light_output:
            display(HTML("<b style='color:red;'>⚠ Could not parse — use e.g. 3-7, 13-17</b>"))
        return
    with light_output:
        display(HTML(f"<b>✓ Light periods:</b> {sorted(light_periods_set)}<br>All other periods → Dark."))

set_light_button.on_click(on_set_light)

display(HTML("<h3>Step 3 — Light Periods</h3>"))
display(HTML("<small>Periods when light is <b>ON</b>. Supports ranges (3-7) and lists (3,4,5). Leave blank to skip Light/Dark split.</small>"))
display(widgets.HBox([light_input, set_light_button]))
display(light_output)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Shared: Excluded periods                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
excluded_periods_input = widgets.Text(
    value="1, 2", description="Exclude pn:",
    placeholder="e.g. 1,2",
    layout=widgets.Layout(width="300px"))
set_exclude_button = widgets.Button(description="Set exclusion periods",
                                     layout=widgets.Layout(width="200px"),
                                     button_style="primary")
exclude_output = widgets.Output()

def on_set_exclude(b):
    with exclude_output:
        clear_output()
    txt = excluded_periods_input.value.strip()
    if not txt:
        with exclude_output:
            display(HTML("<b>✓</b> No periods excluded."))
        return
    try:
        excl = parse_period_spec(txt)
        with exclude_output:
            display(HTML(f"<b>✓ Excluding periods:</b> {sorted(excl)}"))
    except Exception:
        with exclude_output:
            display(HTML("<b style='color:red;'>⚠ Could not parse — use e.g. 1,2 or 1-3</b>"))

set_exclude_button.on_click(on_set_exclude)

display(HTML("<h3>Periods to Exclude (both analyses)</h3>"))
display(HTML("<small>e.g. acclimation periods. Pre-filled with <code>1, 2</code> (first 2 min).</small>"))
display(widgets.HBox([excluded_periods_input, set_exclude_button]))
display(exclude_output)


Textarea(value='', description='Or paths:', layout=Layout(height='60px', width='500px'), placeholder='One file…

Text(value='', description='Name:', layout=Layout(width='400px'), placeholder='(defaults to filename(s))')

Output()

VBox()

Button(button_style='primary', description='Save groups', layout=Layout(width='200px'), style=ButtonStyle())

Output()

Output()

Text(value='1, 2', description='Exclude pn:', layout=Layout(width='300px'), placeholder='e.g. 1,2')

---
### Analysis

Each analysis below produces three outputs: per-well scatter plots with group means ± 95% CI, a time-series trace across periods, and a category breakdown showing the composition of movement types (per-fish %, then group-averaged). Results are exported to an Excel file.


In [4]:
# @title 3 · Distance Analysis

def run_analysis(measure_name, unit, column_prefix, checkboxes, output_widget):
    with output_widget:
        clear_output()

        if df is None:
            display(HTML("<b style='color:red;'>⚠ No data loaded. Run cell 2 first.</b>"))
            return
        if not well_group_mapping:
            display(HTML("<b style='color:red;'>⚠ No groups defined. Run cell 2 first.</b>"))
            return

        selected_cols = []
        for key, cb in checkboxes.items():
            col = key + column_prefix
            if cb.value:
                if col not in df.columns:
                    display(HTML(f"<b style='color:red;'>⚠ Column '{col}' not in spreadsheet.</b>"))
                    return
                selected_cols.append(col)
        if not selected_cols:
            display(HTML(f"<b style='color:red;'>⚠ No {measure_name.lower()} columns selected.</b>"))
            return

        display(HTML(f"<b>Summing:</b> {' + '.join(selected_cols)}"))

        dfa = df.copy()
        dfa['total_val'] = dfa[selected_cols].sum(axis=1)
        dfa['Group'] = dfa['aname'].apply(well_to_group)

        if light_periods_set:
            dfa['Phase'] = np.where(dfa['pn'].isin(light_periods_set), 'Light', 'Dark')
        else:
            dfa['Phase'] = 'Overall'

        if excluded_periods_input.value.strip():
            try:
                excl = parse_period_spec(excluded_periods_input.value)
                n_before = len(dfa)
                dfa = dfa[~dfa['pn'].isin(excl)]
                display(HTML(f"<small>Excluded pn {sorted(excl)} ({n_before - len(dfa)} rows removed)</small>"))
            except Exception:
                display(HTML("<b style='color:red;'>⚠ Bad exclude-period format.</b>"))
                return

        dfa = dfa.dropna(subset=['Group'])
        if dfa.empty:
            display(HTML("<b style='color:red;'>⚠ No data after filtering — check groups.</b>"))
            return

        n_groups = dfa['Group'].nunique()
        n_wells = dfa['aname'].nunique()
        display(HTML(f"<small>{n_wells} wells in {n_groups} groups, {dfa['pn'].nunique()} periods</small>"))

        groups = sorted(dfa['Group'].unique())
        cmap = plt.get_cmap('tab10', max(len(groups), 1))
        gcols = {g: cmap(i) for i, g in enumerate(groups)}

        # ══════════════════════════════════════════════════════════════
        # 1. TIME SERIES (always shown first)
        # ══════════════════════════════════════════════════════════════
        ts = dfa.groupby(['Group','pn']).agg(
            mean_val=('total_val','mean'),
            std_val=('total_val','std'),
            n=('total_val','count')
        ).reset_index()
        ts['se'] = ts['std_val'] / np.sqrt(ts['n'])
        ts['ci95'] = ts.apply(
            lambda r: stats.t.ppf(0.975, max(r['n']-1, 1)) * r['se'] if r['n'] > 1 else 0,
            axis=1)

        fig, ax = plt.subplots(figsize=(10, 5))
        for g in sorted(ts['Group'].unique()):
            gd = ts[ts['Group']==g].sort_values('pn')
            ax.errorbar(gd['pn'], gd['mean_val'], yerr=gd['ci95'],
                        fmt='-o', label=g, color=gcols.get(g), capsize=3, markersize=4)
        if light_periods_set:
            for pn_val in sorted(light_periods_set):
                ax.axvspan(pn_val - 0.5, pn_val + 0.5, color='yellow', alpha=0.15)
        ax.set_xlabel("Period (pn)")
        ax.set_ylabel(f"Mean {measure_name} ({unit})")
        ax.set_title(f"Time Series — {measure_name} ± 95% CI (t)\n{spreadsheet_name}", fontsize=13)
        ax.legend(title="Group", fontsize=9)
        plt.tight_layout()
        plt.show()

        # ══════════════════════════════════════════════════════════════
        # 2. PRE-COMPUTE per-well aggregates for each phase
        # ══════════════════════════════════════════════════════════════
        phase_options = ['Overall']
        if light_periods_set:
            phase_options += ['Light', 'Dark']

        # Per-well totals by phase
        agg_by_phase = {}
        for phase in phase_options:
            if phase == 'Overall':
                sub = dfa
            elif phase == 'Light':
                sub = dfa[dfa['Phase'] == 'Light']
            else:
                sub = dfa[dfa['Phase'] == 'Dark']
            pw = sub.groupby('aname')['total_val'].sum().reset_index().rename(columns={'total_val': phase})
            pw['Group'] = pw['aname'].apply(well_to_group)
            pw = pw.dropna(subset=['Group'])
            agg_by_phase[phase] = pw

        # Merge all phases into one agg table for export
        agg = agg_by_phase['Overall'].copy()
        for phase in phase_options[1:]:
            agg = agg.merge(agg_by_phase[phase][['aname', phase]], on='aname', how='left')
            agg[phase] = agg[phase].fillna(0)

        # ══════════════════════════════════════════════════════════════
        # 3. INTERACTIVE: Per-well scatter + summary
        # ══════════════════════════════════════════════════════════════
        scatter_out = widgets.Output()

        def draw_scatter(phase):
            with scatter_out:
                clear_output(wait=True)
                pw = agg_by_phase[phase]
                group_pos = {g: i for i, g in enumerate(groups)}

                fig, ax = plt.subplots(figsize=(6, 5.5))
                for _, row in pw.iterrows():
                    g = row['Group']
                    jitter = np.random.uniform(-0.15, 0.15)
                    ax.scatter(group_pos[g] + jitter, row[phase],
                               color=gcols[g], alpha=0.7, s=50, zorder=3)
                for g in groups:
                    vals = pw.loc[pw['Group']==g, phase]
                    m = vals.mean()
                    ci = ci95_t(vals)
                    ax.errorbar(group_pos[g], m, yerr=ci, fmt='o',
                                color='black', capsize=5, markersize=10, zorder=4)
                ax.set_xticks(list(group_pos.values()))
                ax.set_xticklabels(list(group_pos.keys()))
                ax.set_xlim(-0.5, len(groups) - 0.5)
                ax.set_xlabel("Group")
                ax.set_ylabel(f"Total {measure_name} ({unit})")
                ax.set_title(f"{phase} {measure_name} — Per Well ± 95% CI (t)\n{spreadsheet_name}", fontsize=13)
                plt.tight_layout()
                plt.show()

                # Summary table
                rows = []
                for g in groups:
                    vals = pw.loc[pw['Group']==g, phase]
                    rows.append({'Group': g, 'Mean': vals.mean(), 'n': int(vals.count()),
                                 '95% CI': ci95_t(vals)})
                summary = pd.DataFrame(rows)
                try:
                    display(summary.style.format({'Mean': '{:.1f}', '95% CI': '{:.1f}'}).hide(axis='index'))
                except Exception:
                    display(summary.round(1))

        scatter_dropdown = widgets.Dropdown(
            options=phase_options, value='Overall',
            description='Phase:', layout=widgets.Layout(width='200px'))
        scatter_dropdown.observe(lambda change: draw_scatter(change['new']), names='value')

        display(HTML(f"<h4>Per-well {measure_name}</h4>"))
        display(scatter_dropdown)
        display(scatter_out)
        draw_scatter('Overall')

        # ══════════════════════════════════════════════════════════════
        # 4. INTERACTIVE: Category breakdown — grouped bars with CI
        # ══════════════════════════════════════════════════════════════
        cat_cols = {
            'Large': 'lar' + column_prefix,
            'Small': 'sml' + column_prefix,
            'Inactive': 'ina' + column_prefix,
        }
        cat_cols = {k: v for k, v in cat_cols.items() if v in dfa.columns}

        cat_well_export = None
        all_cat_summaries = []

        if len(cat_cols) >= 2:
            cat_names = list(cat_cols.keys())
            bar_colors = {'Large': '#e74c3c', 'Small': '#3498db', 'Inactive': '#95a5a6'}

            # Pre-compute per-fish data for each phase
            cat_data_by_phase = {}
            cat_well_all_phases = []
            for phase in phase_options:
                if phase == 'Overall':
                    sub = dfa
                elif phase == 'Light':
                    sub = dfa[dfa['Phase'] == 'Light']
                else:
                    sub = dfa[dfa['Phase'] == 'Dark']

                pw = sub.groupby('aname')[list(cat_cols.values())].sum().reset_index()
                pw.rename(columns={v: k for k, v in cat_cols.items()}, inplace=True)
                pw['Group'] = pw['aname'].apply(well_to_group)
                pw = pw.dropna(subset=['Group'])

                # Per-fish percentages
                pw['_total'] = pw[cat_names].sum(axis=1)
                for _c in cat_names:
                    pw[f'{_c} %'] = np.where(pw['_total'] > 0, pw[_c] / pw['_total'] * 100, 0)

                cat_data_by_phase[phase] = pw

                # Store for export (with % columns, tagged by phase)
                pw_export = pw.copy()
                pw_export.insert(0, 'Phase', phase)
                cat_well_all_phases.append(pw_export)

                # Build summary row for export
                cs_rows = []
                for g in groups:
                    gd = pw[pw['Group'] == g]
                    row = {'Phase': phase, 'Group': g}
                    for _c in cat_names:
                        row[_c] = gd[_c].mean()
                        row[f'{_c} %'] = gd[f'{_c} %'].mean()
                    row['Total'] = gd[cat_names].sum(axis=1).mean()
                    cs_rows.append(row)
                all_cat_summaries.extend(cs_rows)

            cat_out = widgets.Output()

            def draw_categories(phase):
                with cat_out:
                    clear_output(wait=True)
                    pw = cat_data_by_phase[phase]
                    n_cats = len(cat_names)
                    n_groups = len(groups)

                    # ── Grouped bar: per-fish percentages with CI ──
                    fig, ax = plt.subplots(figsize=(max(7, n_groups * 2.5), 5))
                    x = np.arange(n_groups)
                    total_bar_width = 0.7
                    w = total_bar_width / n_cats

                    for ci, _c in enumerate(cat_names):
                        means = [pw.loc[pw['Group']==g, f'{_c} %'].mean() for g in groups]
                        cis = [ci95_t(pw.loc[pw['Group']==g, f'{_c} %']) for g in groups]
                        offset = (ci - (n_cats - 1) / 2) * w
                        ax.bar(x + offset, means, w * 0.9, yerr=cis,
                               color=bar_colors.get(_c, '#999'), label=_c,
                               capsize=3, error_kw={'lw': 1}, alpha=0.85)
                    ax.set_xticks(x)
                    ax.set_xticklabels(groups)
                    ax.set_ylabel(f"% of Total {measure_name}")
                    ax.set_title(f"{phase} Category Breakdown — Per-fish % ± 95% CI\n{spreadsheet_name}", fontsize=13)
                    ax.legend(title="Category", fontsize=9)
                    ax.set_ylim(0, max(ax.get_ylim()[1], 80))
                    plt.tight_layout()
                    plt.show()

                    # Summary table
                    rows = []
                    for g in groups:
                        gd = pw[pw['Group'] == g]
                        row = {'Group': g}
                        for _c in cat_names:
                            row[_c] = f"{gd[_c].mean():.1f}"
                            row[f'{_c} %'] = f"{gd[f'{_c} %'].mean():.1f}"
                        row['Total'] = f"{gd[cat_names].sum(axis=1).mean():.1f}"
                        rows.append(row)
                    display(pd.DataFrame(rows))

            cat_dropdown = widgets.Dropdown(
                options=phase_options, value='Overall',
                description='Phase:', layout=widgets.Layout(width='200px'))
            cat_dropdown.observe(lambda change: draw_categories(change['new']), names='value')

            display(HTML(f"<h4>{measure_name} Category Breakdown</h4>"))
            display(cat_dropdown)
            display(cat_out)
            draw_categories('Overall')

            cat_summary_df = pd.DataFrame(all_cat_summaries)
        else:
            cat_well_all_phases = []
            cat_summary_df = None

        # ══════════════════════════════════════════════════════════════
        # 5. EXCEL EXPORT
        # ══════════════════════════════════════════════════════════════
        safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in spreadsheet_name)
        out_file = f"{safe}_{measure_name.lower()}_results.xlsx"
        with pd.ExcelWriter(out_file) as writer:
            # Per-well totals (all plates combined)
            agg.to_excel(writer, sheet_name="Per-well", index=False)

            # Time series (all plates combined)
            ts.to_excel(writer, sheet_name="Time Series", index=False)

            # Raw source data — one tab per plate
            if plate_labels:
                for plabel in plate_labels:
                    raw = df[df['aname'].str.startswith(plabel + ':')]
                    if not raw.empty:
                        raw.to_excel(writer, sheet_name=f"{plabel} Raw", index=False)
            else:
                df.to_excel(writer, sheet_name="Raw", index=False)

            # Category breakdown — all phases, per fish, with % columns
            if cat_well_all_phases:
                cat_export = pd.concat(cat_well_all_phases, ignore_index=True)
                # Drop internal _total column
                cat_export = cat_export.drop(columns=['_total'], errors='ignore')
                cat_export.to_excel(writer, sheet_name="Category Per-fish", index=False)

            # Category summary — group means of per-fish values
            if cat_summary_df is not None:
                cat_summary_df.to_excel(writer, sheet_name="Category Group Means", index=False)

        display(HTML(f"<h4>📥 {out_file}</h4>"))
        if _COLAB:
            colab_files.download(out_file)
        elif _FILELINK:
            display(FileLink(out_file))
        else:
            display(HTML(f"<p>Saved: <code>{out_file}</code></p>"))


if 'inc_lardist' not in dir():
    inc_lardist = widgets.Checkbox(value=True,  description='Large distance (lardist)')
    inc_smldist = widgets.Checkbox(value=True,  description='Small distance (smldist)')
    inc_inadist = widgets.Checkbox(value=False, description='Inactive distance (inadist)')

dist_button = widgets.Button(description='Run Distance Analysis',
                              layout=widgets.Layout(width='250px'), button_style='primary')
dist_output = widgets.Output()

def on_dist(b):
    run_analysis('Movement', 'mm', 'dist',
                 {'lar': inc_lardist, 'sml': inc_smldist, 'ina': inc_inadist},
                 dist_output)

dist_button.on_click(on_dist)

display(HTML("<h3>Distance Analysis</h3>"))
display(HTML("<small><b>lardist</b> = large movements &ensp;|&ensp; <b>smldist</b> = small movements &ensp;|&ensp; <b>inadist</b> = drift during inactivity (usually excluded)</small>"))
display(inc_lardist, inc_smldist, inc_inadist)
display(dist_button, dist_output)


Checkbox(value=True, description='Large distance (lardist)')

Checkbox(value=True, description='Small distance (smldist)')

Checkbox(value=False, description='Inactive distance (inadist)')

Button(button_style='primary', description='Run Distance Analysis', layout=Layout(width='250px'), style=Button…

Output()

In [ ]:
# @title 4 · Duration Analysis

def run_analysis(measure_name, unit, column_prefix, checkboxes, output_widget):
    with output_widget:
        clear_output()

        if df is None:
            display(HTML("<b style='color:red;'>⚠ No data loaded. Run cell 2 first.</b>"))
            return
        if not well_group_mapping:
            display(HTML("<b style='color:red;'>⚠ No groups defined. Run cell 2 first.</b>"))
            return

        selected_cols = []
        for key, cb in checkboxes.items():
            col = key + column_prefix
            if cb.value:
                if col not in df.columns:
                    display(HTML(f"<b style='color:red;'>⚠ Column '{col}' not in spreadsheet.</b>"))
                    return
                selected_cols.append(col)
        if not selected_cols:
            display(HTML(f"<b style='color:red;'>⚠ No {measure_name.lower()} columns selected.</b>"))
            return

        display(HTML(f"<b>Summing:</b> {' + '.join(selected_cols)}"))

        dfa = df.copy()
        dfa['total_val'] = dfa[selected_cols].sum(axis=1)
        dfa['Group'] = dfa['aname'].apply(well_to_group)

        if light_periods_set:
            dfa['Phase'] = np.where(dfa['pn'].isin(light_periods_set), 'Light', 'Dark')
        else:
            dfa['Phase'] = 'Overall'

        if excluded_periods_input.value.strip():
            try:
                excl = parse_period_spec(excluded_periods_input.value)
                n_before = len(dfa)
                dfa = dfa[~dfa['pn'].isin(excl)]
                display(HTML(f"<small>Excluded pn {sorted(excl)} ({n_before - len(dfa)} rows removed)</small>"))
            except Exception:
                display(HTML("<b style='color:red;'>⚠ Bad exclude-period format.</b>"))
                return

        dfa = dfa.dropna(subset=['Group'])
        if dfa.empty:
            display(HTML("<b style='color:red;'>⚠ No data after filtering — check groups.</b>"))
            return

        n_groups = dfa['Group'].nunique()
        n_wells = dfa['aname'].nunique()
        display(HTML(f"<small>{n_wells} wells in {n_groups} groups, {dfa['pn'].nunique()} periods</small>"))

        groups = sorted(dfa['Group'].unique())
        cmap = plt.get_cmap('tab10', max(len(groups), 1))
        gcols = {g: cmap(i) for i, g in enumerate(groups)}

        # ══════════════════════════════════════════════════════════════
        # 1. TIME SERIES (always shown first)
        # ══════════════════════════════════════════════════════════════
        ts = dfa.groupby(['Group','pn']).agg(
            mean_val=('total_val','mean'),
            std_val=('total_val','std'),
            n=('total_val','count')
        ).reset_index()
        ts['se'] = ts['std_val'] / np.sqrt(ts['n'])
        ts['ci95'] = ts.apply(
            lambda r: stats.t.ppf(0.975, max(r['n']-1, 1)) * r['se'] if r['n'] > 1 else 0,
            axis=1)

        fig, ax = plt.subplots(figsize=(10, 5))
        for g in sorted(ts['Group'].unique()):
            gd = ts[ts['Group']==g].sort_values('pn')
            ax.errorbar(gd['pn'], gd['mean_val'], yerr=gd['ci95'],
                        fmt='-o', label=g, color=gcols.get(g), capsize=3, markersize=4)
        if light_periods_set:
            for pn_val in sorted(light_periods_set):
                ax.axvspan(pn_val - 0.5, pn_val + 0.5, color='yellow', alpha=0.15)
        ax.set_xlabel("Period (pn)")
        ax.set_ylabel(f"Mean {measure_name} ({unit})")
        ax.set_title(f"Time Series — {measure_name} ± 95% CI (t)\n{spreadsheet_name}", fontsize=13)
        ax.legend(title="Group", fontsize=9)
        plt.tight_layout()
        plt.show()

        # ══════════════════════════════════════════════════════════════
        # 2. PRE-COMPUTE per-well aggregates for each phase
        # ══════════════════════════════════════════════════════════════
        phase_options = ['Overall']
        if light_periods_set:
            phase_options += ['Light', 'Dark']

        # Per-well totals by phase
        agg_by_phase = {}
        for phase in phase_options:
            if phase == 'Overall':
                sub = dfa
            elif phase == 'Light':
                sub = dfa[dfa['Phase'] == 'Light']
            else:
                sub = dfa[dfa['Phase'] == 'Dark']
            pw = sub.groupby('aname')['total_val'].sum().reset_index().rename(columns={'total_val': phase})
            pw['Group'] = pw['aname'].apply(well_to_group)
            pw = pw.dropna(subset=['Group'])
            agg_by_phase[phase] = pw

        # Merge all phases into one agg table for export
        agg = agg_by_phase['Overall'].copy()
        for phase in phase_options[1:]:
            agg = agg.merge(agg_by_phase[phase][['aname', phase]], on='aname', how='left')
            agg[phase] = agg[phase].fillna(0)

        # ══════════════════════════════════════════════════════════════
        # 3. INTERACTIVE: Per-well scatter + summary
        # ══════════════════════════════════════════════════════════════
        scatter_out = widgets.Output()

        def draw_scatter(phase):
            with scatter_out:
                clear_output(wait=True)
                pw = agg_by_phase[phase]
                group_pos = {g: i for i, g in enumerate(groups)}

                fig, ax = plt.subplots(figsize=(6, 5.5))
                for _, row in pw.iterrows():
                    g = row['Group']
                    jitter = np.random.uniform(-0.15, 0.15)
                    ax.scatter(group_pos[g] + jitter, row[phase],
                               color=gcols[g], alpha=0.7, s=50, zorder=3)
                for g in groups:
                    vals = pw.loc[pw['Group']==g, phase]
                    m = vals.mean()
                    ci = ci95_t(vals)
                    ax.errorbar(group_pos[g], m, yerr=ci, fmt='o',
                                color='black', capsize=5, markersize=10, zorder=4)
                ax.set_xticks(list(group_pos.values()))
                ax.set_xticklabels(list(group_pos.keys()))
                ax.set_xlim(-0.5, len(groups) - 0.5)
                ax.set_xlabel("Group")
                ax.set_ylabel(f"Total {measure_name} ({unit})")
                ax.set_title(f"{phase} {measure_name} — Per Well ± 95% CI (t)\n{spreadsheet_name}", fontsize=13)
                plt.tight_layout()
                plt.show()

                # Summary table
                rows = []
                for g in groups:
                    vals = pw.loc[pw['Group']==g, phase]
                    rows.append({'Group': g, 'Mean': vals.mean(), 'n': int(vals.count()),
                                 '95% CI': ci95_t(vals)})
                summary = pd.DataFrame(rows)
                try:
                    display(summary.style.format({'Mean': '{:.1f}', '95% CI': '{:.1f}'}).hide(axis='index'))
                except Exception:
                    display(summary.round(1))

        scatter_dropdown = widgets.Dropdown(
            options=phase_options, value='Overall',
            description='Phase:', layout=widgets.Layout(width='200px'))
        scatter_dropdown.observe(lambda change: draw_scatter(change['new']), names='value')

        display(HTML(f"<h4>Per-well {measure_name}</h4>"))
        display(scatter_dropdown)
        display(scatter_out)
        draw_scatter('Overall')

        # ══════════════════════════════════════════════════════════════
        # 4. INTERACTIVE: Category breakdown — grouped bars with CI
        # ══════════════════════════════════════════════════════════════
        cat_cols = {
            'Large': 'lar' + column_prefix,
            'Small': 'sml' + column_prefix,
            'Inactive': 'ina' + column_prefix,
        }
        cat_cols = {k: v for k, v in cat_cols.items() if v in dfa.columns}

        cat_well_export = None
        all_cat_summaries = []

        if len(cat_cols) >= 2:
            cat_names = list(cat_cols.keys())
            bar_colors = {'Large': '#e74c3c', 'Small': '#3498db', 'Inactive': '#95a5a6'}

            # Pre-compute per-fish data for each phase
            cat_data_by_phase = {}
            cat_well_all_phases = []
            for phase in phase_options:
                if phase == 'Overall':
                    sub = dfa
                elif phase == 'Light':
                    sub = dfa[dfa['Phase'] == 'Light']
                else:
                    sub = dfa[dfa['Phase'] == 'Dark']

                pw = sub.groupby('aname')[list(cat_cols.values())].sum().reset_index()
                pw.rename(columns={v: k for k, v in cat_cols.items()}, inplace=True)
                pw['Group'] = pw['aname'].apply(well_to_group)
                pw = pw.dropna(subset=['Group'])

                # Per-fish percentages
                pw['_total'] = pw[cat_names].sum(axis=1)
                for _c in cat_names:
                    pw[f'{_c} %'] = np.where(pw['_total'] > 0, pw[_c] / pw['_total'] * 100, 0)

                cat_data_by_phase[phase] = pw

                # Store for export (with % columns, tagged by phase)
                pw_export = pw.copy()
                pw_export.insert(0, 'Phase', phase)
                cat_well_all_phases.append(pw_export)

                # Build summary row for export
                cs_rows = []
                for g in groups:
                    gd = pw[pw['Group'] == g]
                    row = {'Phase': phase, 'Group': g}
                    for _c in cat_names:
                        row[_c] = gd[_c].mean()
                        row[f'{_c} %'] = gd[f'{_c} %'].mean()
                    row['Total'] = gd[cat_names].sum(axis=1).mean()
                    cs_rows.append(row)
                all_cat_summaries.extend(cs_rows)

            cat_out = widgets.Output()

            def draw_categories(phase):
                with cat_out:
                    clear_output(wait=True)
                    pw = cat_data_by_phase[phase]
                    n_cats = len(cat_names)
                    n_groups = len(groups)

                    # ── Grouped bar: per-fish percentages with CI ──
                    fig, ax = plt.subplots(figsize=(max(7, n_groups * 2.5), 5))
                    x = np.arange(n_groups)
                    total_bar_width = 0.7
                    w = total_bar_width / n_cats

                    for ci, _c in enumerate(cat_names):
                        means = [pw.loc[pw['Group']==g, f'{_c} %'].mean() for g in groups]
                        cis = [ci95_t(pw.loc[pw['Group']==g, f'{_c} %']) for g in groups]
                        offset = (ci - (n_cats - 1) / 2) * w
                        ax.bar(x + offset, means, w * 0.9, yerr=cis,
                               color=bar_colors.get(_c, '#999'), label=_c,
                               capsize=3, error_kw={'lw': 1}, alpha=0.85)
                    ax.set_xticks(x)
                    ax.set_xticklabels(groups)
                    ax.set_ylabel(f"% of Total {measure_name}")
                    ax.set_title(f"{phase} Category Breakdown — Per-fish % ± 95% CI\n{spreadsheet_name}", fontsize=13)
                    ax.legend(title="Category", fontsize=9)
                    ax.set_ylim(0, max(ax.get_ylim()[1], 80))
                    plt.tight_layout()
                    plt.show()

                    # Summary table
                    rows = []
                    for g in groups:
                        gd = pw[pw['Group'] == g]
                        row = {'Group': g}
                        for _c in cat_names:
                            row[_c] = f"{gd[_c].mean():.1f}"
                            row[f'{_c} %'] = f"{gd[f'{_c} %'].mean():.1f}"
                        row['Total'] = f"{gd[cat_names].sum(axis=1).mean():.1f}"
                        rows.append(row)
                    display(pd.DataFrame(rows))

            cat_dropdown = widgets.Dropdown(
                options=phase_options, value='Overall',
                description='Phase:', layout=widgets.Layout(width='200px'))
            cat_dropdown.observe(lambda change: draw_categories(change['new']), names='value')

            display(HTML(f"<h4>{measure_name} Category Breakdown</h4>"))
            display(cat_dropdown)
            display(cat_out)
            draw_categories('Overall')

            cat_summary_df = pd.DataFrame(all_cat_summaries)
        else:
            cat_well_all_phases = []
            cat_summary_df = None

        # ══════════════════════════════════════════════════════════════
        # 5. EXCEL EXPORT
        # ══════════════════════════════════════════════════════════════
        safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in spreadsheet_name)
        out_file = f"{safe}_{measure_name.lower()}_results.xlsx"
        with pd.ExcelWriter(out_file) as writer:
            # Per-well totals (all plates combined)
            agg.to_excel(writer, sheet_name="Per-well", index=False)

            # Time series (all plates combined)
            ts.to_excel(writer, sheet_name="Time Series", index=False)

            # Raw source data — one tab per plate
            if plate_labels:
                for plabel in plate_labels:
                    raw = df[df['aname'].str.startswith(plabel + ':')]
                    if not raw.empty:
                        raw.to_excel(writer, sheet_name=f"{plabel} Raw", index=False)
            else:
                df.to_excel(writer, sheet_name="Raw", index=False)

            # Category breakdown — all phases, per fish, with % columns
            if cat_well_all_phases:
                cat_export = pd.concat(cat_well_all_phases, ignore_index=True)
                # Drop internal _total column
                cat_export = cat_export.drop(columns=['_total'], errors='ignore')
                cat_export.to_excel(writer, sheet_name="Category Per-fish", index=False)

            # Category summary — group means of per-fish values
            if cat_summary_df is not None:
                cat_summary_df.to_excel(writer, sheet_name="Category Group Means", index=False)

        display(HTML(f"<h4>📥 {out_file}</h4>"))
        if _COLAB:
            colab_files.download(out_file)
        elif _FILELINK:
            display(FileLink(out_file))
        else:
            display(HTML(f"<p>Saved: <code>{out_file}</code></p>"))


if 'inc_lardur' not in dir():
    inc_lardur = widgets.Checkbox(value=True,  description='Large duration (lardur)')
    inc_smldur = widgets.Checkbox(value=True,  description='Small duration (smldur)')
    inc_inadur = widgets.Checkbox(value=False, description='Inactive duration (inadur)')

dur_button = widgets.Button(description='Run Duration Analysis',
                             layout=widgets.Layout(width='250px'), button_style='primary')
dur_output = widgets.Output()

def on_dur(b):
    run_analysis('Duration', 'sec', 'dur',
                 {'lar': inc_lardur, 'sml': inc_smldur, 'ina': inc_inadur},
                 dur_output)

dur_button.on_click(on_dur)

display(HTML("<h3>Duration Analysis</h3>"))
display(HTML("<small><b>lardur</b> = time in large movement &ensp;|&ensp; <b>smldur</b> = time in small movement &ensp;|&ensp; <b>inadur</b> = time inactive (complement of activity, usually excluded)</small>"))
display(inc_lardur, inc_smldur, inc_inadur)
display(dur_button, dur_output)
